 # Perturbation Analysis



 scHopfield supports two complementary strategies for in-silico perturbations:



 **ODE-based** — integrates the full dynamical system

 $$\frac{dx}{dt} = W \cdot \sigma(x) - \gamma \cdot x + I$$

 with a fixed perturbation (e.g. GATA1 = 0 for knockout).



 **GRN propagation** — propagates expression shifts through the learned network

 without time integration, analogous to the CellOracle approach.



 Topics covered:

 1. ODE-based single-cell trajectories

 2. ODE-based gene perturbation (single cell)

 3. Population-level cell fate shift (ODE)

 4. GRN propagation perturbations

 5. Perturbation effect scoring

 6. Comparing multiple perturbations side-by-side

 ## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scHopfield as sch

DATA_PATH    = '/path/to/data/'
DATASET_FILE = 'hematopoiesis.h5ad'
MODEL_FILE   = 'model.h5sch'

CLUSTER_KEY  = 'cell_type'
SPLICED_KEY  = 'M_t'
BASIS        = 'umap'

CELL_TYPE_ORDER = ['Meg', 'Ery', 'MEP-like', 'HSC', 'GMP-like', 'Mon', 'Bas', 'Neu']

# Gene of interest for perturbation
GOI = 'GATA1'

adata = sc.read_h5ad(DATA_PATH + DATASET_FILE)
sch.tl.load_model(adata, MODEL_FILE)
print(adata)

colors = {ct: f'C{i}' for i, ct in enumerate(CELL_TYPE_ORDER)}


 ## 5.1 ODE-based Single-Cell Trajectory



 Simulate the continuous relaxation of a single cell's expression state under

 the scHopfield dynamics.

In [ ]:
TARGET_CLUSTER = 'HSC'
T_SPAN = np.linspace(0, 10, 1000)

# Pick a representative cell from the target cluster
cluster_mask    = adata.obs[CLUSTER_KEY] == TARGET_CLUSTER
cluster_indices = np.where(cluster_mask)[0]
cell_idx        = cluster_indices[len(cluster_indices) // 2]

wt_trajectory = sch.dyn.simulate_trajectory(
    adata,
    cluster=TARGET_CLUSTER,
    spliced_key=SPLICED_KEY,
    cell_idx=cell_idx,
    t_span=T_SPAN
)

print(f"Trajectory shape: {wt_trajectory.shape}  (time steps × genes)")


In [ ]:
# Identify top dynamically changing genes (WT)
delta_wt = np.abs(wt_trajectory[-1] - wt_trajectory[0])
top_idx  = np.argsort(delta_wt)[-6:]

fig, ax = plt.subplots(figsize=(8, 5), tight_layout=True)
for i, idx in enumerate(top_idx):
    ax.plot(T_SPAN, wt_trajectory[:, idx],
            label=adata.var_names[idx], linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Expression')
ax.set_title(f'{TARGET_CLUSTER}: WT trajectory (top dynamic genes)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()


 ## 5.2 ODE-based Gene Perturbation (single cell)



 Fix the perturbed gene at a constant value and simulate the response.

In [ ]:
ko_trajectory = sch.dyn.simulate_perturbation_ode(
    adata,
    cluster=TARGET_CLUSTER,
    spliced_key=SPLICED_KEY,
    cell_idx=cell_idx,
    gene_perturbations={GOI: 0.0},
    t_span=T_SPAN
)

oe_trajectory = sch.dyn.simulate_perturbation_ode(
    adata,
    cluster=TARGET_CLUSTER,
    spliced_key=SPLICED_KEY,
    cell_idx=cell_idx,
    gene_perturbations={GOI: 10.0},
    t_span=T_SPAN
)


In [ ]:
# Identify most responsive genes (WT vs KO endpoint)
diff     = np.abs(ko_trajectory[-1] - wt_trajectory[-1])
top_resp = np.argsort(diff)[-5:]
plot_colors = plt.cm.tab10(np.linspace(0, 1, len(top_resp)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)

# Panel 1: WT vs KO trajectories
for i, idx in enumerate(top_resp):
    gene = adata.var_names[idx]
    axes[0].plot(T_SPAN, wt_trajectory[:, idx], '-',  color=plot_colors[i], linewidth=2, label=gene)
    axes[0].plot(T_SPAN, ko_trajectory[:, idx], '--', color=plot_colors[i], linewidth=2)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Expression')
axes[0].set_title(f'{TARGET_CLUSTER}: Top responsive genes ({GOI} KO)\nSolid=WT, Dashed=KO')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Panel 2: Phase-space divergence
dist_wt = np.linalg.norm(wt_trajectory - wt_trajectory[0], axis=1)
dist_ko = np.linalg.norm(ko_trajectory - wt_trajectory[0], axis=1)
dist_oe = np.linalg.norm(oe_trajectory - wt_trajectory[0], axis=1)

axes[1].plot(T_SPAN, dist_wt, label='WT',        color='gray',    linewidth=2)
axes[1].plot(T_SPAN, dist_ko, label=f'{GOI} KO', color='#E74C3C', linewidth=2)
axes[1].plot(T_SPAN, dist_oe, label=f'{GOI} OE', color='#3498DB', linewidth=2)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Distance from origin state')
axes[1].set_title(f'{TARGET_CLUSTER}: Trajectory divergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.show()


 ## 5.3 Population-level Cell Fate Shift (ODE)



 Run the ODE simulation on all cells simultaneously to obtain a perturbed

 expression landscape.  The result can be projected back onto the UMAP.

In [ ]:
print(f"Running dataset-wide ODE simulation for {GOI} KO...")

adata_ode_ko = sch.dyn.simulate_shift_ode(
    adata.copy(),
    perturb_condition={GOI: 0.0},
    cluster_key=CLUSTER_KEY,
    dt=5.0,
    use_cluster_specific_GRN=True,
    verbose=True,
    n_jobs=-1,
)

print(f"Running dataset-wide ODE simulation for {GOI} OE...")

adata_ode_oe = sch.dyn.simulate_shift_ode(
    adata.copy(),
    perturb_condition={GOI: 10.0},
    cluster_key=CLUSTER_KEY,
    dt=5.0,
    use_cluster_specific_GRN=True,
    verbose=True,
    n_jobs=-1,
)


In [ ]:
# Top affected genes
top_genes_ko = sch.dyn.get_top_affected_genes(adata_ode_ko, n_genes=20)
top_genes_oe = sch.dyn.get_top_affected_genes(adata_ode_oe, n_genes=20)

print(f"Top {GOI} KO-affected genes: {top_genes_ko[:10]}")
print(f"Top {GOI} OE-affected genes: {top_genes_oe[:10]}")


In [ ]:
# Compute Hopfield-based flow for ODE-shifted data
sch.tl.calculate_flow(
    adata_ode_ko,
    basis=BASIS,
    method='hopfield',
    cluster_key=CLUSTER_KEY,
    use_cluster_specific=True,
    source='delta',
    store_key='delta_velocity_hopfield',
    n_neighbors=30,
    n_jobs=4,
    verbose=True
)

sch.tl.calculate_inner_product(
    adata_ode_ko,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2='delta_velocity_hopfield',
    store_key='delta_inner_product_hopfield'
)


In [ ]:
# Visualise KO perturbation flow on embedding
fig, axes = plt.subplots(1, 2, figsize=(14, 6), tight_layout=True)

sch.pl.plot_flow(
    adata_ode_ko, basis=BASIS, ax=axes[0], on_grid=True,
    flow_key='delta_velocity_hopfield',
    n_grid=25, min_mass=25, scale=5000, color='#E74C3C',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} KO — Hopfield Δv (grid)'
)

sch.pl.plot_inner_product(
    adata_ode_ko, basis=BASIS, ax=axes[1],
    inner_product_key='delta_inner_product_hopfield',
    title=f'{GOI} KO — Δv · v₀ inner product'
)

plt.show()


 ## 5.4 GRN Propagation Perturbations



 The propagation approach shifts expression through the network iteratively

 (analogous to CellOracle's approach).

In [ ]:
N_PROPAGATION   = 3
DT              = 5.0
PERTURB_KO_VAL  = 0.0
PERTURB_OE_VAL  = 10.0

print(f"Simulating {GOI} KO (GRN propagation)...")
adata_ko = sch.dyn.simulate_shift(
    adata.copy(),
    perturb_condition={GOI: PERTURB_KO_VAL},
    cluster_key=CLUSTER_KEY,
    n_propagation=N_PROPAGATION,
    use_cluster_specific_GRN=True,
    clip_delta_X=True,
    x_max_percentile=90.0,
    verbose=True,
    dt=DT,
)

print(f"Simulating {GOI} OE (GRN propagation)...")
adata_oe = sch.dyn.simulate_shift(
    adata.copy(),
    perturb_condition={GOI: PERTURB_OE_VAL},
    cluster_key=CLUSTER_KEY,
    n_propagation=N_PROPAGATION,
    use_cluster_specific_GRN=True,
    clip_delta_X=True,
    x_max_percentile=90.0,
    verbose=True,
    dt=DT,
)

top_genes_ko_prop = sch.dyn.get_top_affected_genes(adata_ko, n_genes=20)
top_genes_oe_prop = sch.dyn.get_top_affected_genes(adata_oe, n_genes=20)
print(f"Propagation KO top genes: {top_genes_ko_prop[:5]}")
print(f"Propagation OE top genes: {top_genes_oe_prop[:5]}")


In [ ]:
# Perturbation effect heatmaps (per cluster, per gene)
fig_ko = sch.pl.plot_perturbation_effect_heatmap(
    adata_ko, cluster_key=CLUSTER_KEY, n_genes=30
)
plt.suptitle(f'{GOI} KO — perturbation effect heatmap', y=1.01)
plt.show()

fig_oe = sch.pl.plot_perturbation_effect_heatmap(
    adata_oe, cluster_key=CLUSTER_KEY, n_genes=30
)
plt.suptitle(f'{GOI} OE — perturbation effect heatmap', y=1.01)
plt.show()


In [ ]:
# Top affected genes bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 8), tight_layout=True)

sch.pl.plot_top_affected_genes_bar(adata_ko, n_genes=20, ax=axes[0])
axes[0].set_title(f'{GOI} KO: top 20 affected genes')

sch.pl.plot_top_affected_genes_bar(adata_oe, n_genes=20, ax=axes[1])
axes[1].set_title(f'{GOI} OE: top 20 affected genes')

plt.show()


 ## 5.5 Perturbation Effect Scoring

In [ ]:
# Compute CellOracle-style flow for propagation data
n_neighbors_flow = 50

sch.tl.calculate_flow(
    adata_ko,
    basis=BASIS,
    n_neighbors=n_neighbors_flow,
    method='celloracle',
    correlation_mode='sampled',
    sigma_corr=0.05
)
sch.tl.calculate_inner_product(
    adata_ko,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2=f'perturbation_flow_{BASIS}'
)

sch.tl.calculate_flow(
    adata_oe,
    basis=BASIS,
    n_neighbors=n_neighbors_flow,
    method='celloracle',
    correlation_mode='sampled',
    sigma_corr=0.05
)
sch.tl.calculate_inner_product(
    adata_oe,
    flow_key_1=f'velocity_S_{BASIS}',
    flow_key_2=f'perturbation_flow_{BASIS}'
)


In [ ]:
# Flow visualisation (propagation)
fig, axes = plt.subplots(3, 2, figsize=(10, 15), tight_layout=True)

c = [colors.get(cl, 'gray') for cl in adata_ko.obs[CLUSTER_KEY]]
axes[0, 0].scatter(
    adata_ko.obsm[f'X_{BASIS}'][:, 0],
    adata_ko.obsm[f'X_{BASIS}'][:, 1],
    c=c, s=10, alpha=0.7
)
axes[0, 0].set_title('Cell types')
axes[0, 0].axis('off')

sch.pl.plot_flow(
    adata, basis=BASIS, ax=axes[0, 1], on_grid=True,
    flow_key=f'velocity_S_{BASIS}',
    n_grid=25, min_mass=45, scale=1000, color='black',
    cluster_key=CLUSTER_KEY, colors=colors,
    title='Reference velocity (grid)'
)

sch.pl.plot_flow(
    adata_ko, basis=BASIS, ax=axes[1, 0], on_grid=True,
    flow_key=f'perturbation_flow_{BASIS}',
    n_grid=25, min_mass=25, scale=5, color='#E74C3C',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} KO — perturbation flow (grid)'
)

sch.pl.plot_flow(
    adata_oe, basis=BASIS, ax=axes[1, 1], on_grid=True,
    flow_key=f'perturbation_flow_{BASIS}',
    n_grid=25, min_mass=25, scale=5, color='#3498DB',
    cluster_key=CLUSTER_KEY, colors=colors,
    title=f'{GOI} OE — perturbation flow (grid)'
)

sch.pl.plot_inner_product(
    adata_ko, basis=BASIS, ax=axes[2, 0],
    title=f'{GOI} KO — inner product'
)
sch.pl.plot_inner_product(
    adata_oe, basis=BASIS, ax=axes[2, 1],
    title=f'{GOI} OE — inner product'
)

plt.show()


 ## 5.6 Compare Multiple Perturbations

In [ ]:
# Perturb a set of transcription factors and compare their downstream effects
PERTURBATIONS = {
    'GATA1_KO':  {GOI: 0.0},
    'GATA2_KO':  {'GATA2': 0.0},
    'KLF1_OE':   {'KLF1': 10.0},
}

adata_perturbed = {}
for name, condition in PERTURBATIONS.items():
    print(f"Simulating {name}...")
    adata_perturbed[name] = sch.dyn.simulate_shift(
        adata.copy(),
        perturb_condition=condition,
        cluster_key=CLUSTER_KEY,
        n_propagation=N_PROPAGATION,
        use_cluster_specific_GRN=True,
        clip_delta_X=True,
        verbose=False,
        dt=DT,
    )


In [ ]:
# Side-by-side perturbation effect heatmaps
fig, axes = plt.subplots(1, len(PERTURBATIONS), figsize=(8 * len(PERTURBATIONS), 10))

for ax, (name, adata_p) in zip(axes, adata_perturbed.items()):
    sch.pl.plot_perturbation_effect_heatmap(
        adata_p, cluster_key=CLUSTER_KEY, n_genes=20, ax=ax
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()


In [ ]:
# Per-cluster mean expression shift across perturbations
genes_used     = np.where(adata_ko.var['scHopfield_used'])[0]
gene_names_used = adata_ko.var_names[genes_used]

print("\nTop genes by mean |δX| for each perturbation:")
for name, adata_p in adata_perturbed.items():
    delta_X = adata_p.layers['delta_X'][:, genes_used]
    if hasattr(delta_X, 'toarray'):
        delta_X = delta_X.toarray()
    mean_change = np.abs(delta_X).mean(axis=0)
    top_idx = np.argsort(mean_change)[-10:][::-1]
    print(f"\n  {name}: {list(gene_names_used[top_idx])}")
